In [2]:
from cresowlve.utils import read_json

model_results_path = "../experiments/outputs/gemini-2.5-pro/chgk_en_benchmark_eval_reasoning_model_en_s0_gemini-2.5-pro_20260228_163314.json"
benchmark_path = "../experiments/data/task/chgk_en_benchmark.json"
domain_path = "../experiments/data/task/chgk_benchmark_domain.json"
model_results = read_json(model_results_path)
benchmark = read_json(benchmark_path)
domain_results = read_json(domain_path)
reasoning_type_results = read_json("../experiments/data/task/chgk_benchmark_reasoning_types.json")
factual_ids = [s["id"] for s in reasoning_type_results["data"] if "reasoning_type" in s and s["reasoning_type"] == "factual"]
creative_ids = [s["id"] for s in reasoning_type_results["data"] if "reasoning_type" in s and s["reasoning_type"] == "creative"]

for sample in model_results["data"]:
    benchmark_sample = [s for s in benchmark["data"] if s["id"] == sample["id"]][0]
    sample.update(benchmark_sample)

domain_results["data"] = [s for s in domain_results["data"] if s["id"] in creative_ids]

In [23]:
len(set([domain for sample in domain_results["data"] for domain in sample["domains"]]))

502

In [24]:
len(set([domain for sample in domain_results["data"] for domain in sample["grouped_domains"]]))

34

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import Counter

# ── PASTE YOUR DATA HERE ──────────────────────────────────────────────────────
domain_counts = Counter([grouped_domain for s in domain_results["data"] for grouped_domain in s["grouped_domains"]])

domain_name_mapping = {
    "Art History & Visual Culture": "Art & Culture",
    "Earth & Environmental Science": "Earth & Environment",
    "Medicine & Health Sciences": "Medicine & Health",
    "Astronomy & Space Science": "Astronomy",
    "Physical Education & Sports": "Sports",
    "Home Economics & Daily Life": "Home & Daily Life",
    "Science": "Other Sciences",
    "Design & Architecture": "Design & Arch.",
}

label_font_mapping = {
    "Design & Arch.": 8,
    "Art & Culture": 9,
    "Archaeology": 8,
    "Earth & Environment": 8,
    "Business Studies": 8,
    "Communication": 8,
    "Other Sciences": 7,
    "Medicine & Health": 8,
    "Design & Architecture": 8,
    "Law & Criminology": 7,
    "Engineering & Technology": 8,
    "Literature": 12,
    "History": 12
}

# ─────────────────────────────────────────────────────────────────────────────

total = len(domain_results["data"]) # sum(domain_counts.values())
df = pd.DataFrame({
    "Name":  [domain_name_mapping.get(d, d) for d in list(domain_counts.keys())],
    "Value": list(domain_counts.values()),
})
df["Pct"]   = df["Value"] / total * 100
df["Label"] = df.apply(lambda r: f"{r['Name']}  {int(r['Value'])} ({r['Pct']:.1f}%)", axis=1)

N = len(df)

# ── Light pastel palette ──────────────────────────────────────────────────────
hues   = np.linspace(0, 1, N, endpoint=False)
colors = [mcolors.hsv_to_rgb([h, 0.38, 0.96]) for h in hues]

# ── Bar geometry ──────────────────────────────────────────────────────────────
lowerLimit = 30
upperLimit = 100
max_val    = df["Value"].max()
slope      = (upperLimit - lowerLimit) / max_val
heights    = slope * df["Value"] + lowerLimit

# Add a cap:
maxBarHeight = 60  # tweak this value to your liking
heights = np.minimum(heights, maxBarHeight)

width   = 2 * np.pi / N
indexes = list(range(1, N + 1))
angles  = [i * width for i in indexes]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(22, 22), facecolor="white")
ax  = plt.subplot(111, polar=True)
plt.axis("off")

bars = ax.bar(
    x=angles,
    height=heights,
    width=width,
    bottom=lowerLimit,
    linewidth=1.5,
    edgecolor="white",
    color=colors,
)

# ── Labels inside each bar ────────────────────────────────────────────────────
labelPadding = 3

for bar, angle, height, label in zip(bars, angles, heights, df["Label"]):
    rotation = np.rad2deg(angle)

    if angle >= np.pi / 2 and angle < 3 * np.pi / 2:
        alignment = "left"     # flipped: anchor at far end, text grows inward
        rotation  = rotation + 180
    else:
        alignment = "right"    # flipped: anchor at far end, text grows inward

    label_font = 10

    for domain, font_size in label_font_mapping.items():
        if domain in label:
            label_font = font_size
            break

    ax.text(
        x=angle,
        y=lowerLimit + bar.get_height() - labelPadding,  # just inside the tip
        s=label,
        ha=alignment,
        va="center",
        rotation=rotation,
        rotation_mode="anchor",
        fontsize=label_font * 1.5,
        fontweight="bold",
        color="#333333",
    )

# ── Centre total ──────────────────────────────────────────────────────────────
ax.text(0, 0, f"Total\n{total:,}", ha="center", va="center",
        fontsize=20, fontweight="bold", color="#444444")

plt.tight_layout()
# plt.savefig("../figures/domain_dist.pdf",
#             dpi=360, bbox_inches="tight", facecolor="white")
plt.savefig("../figures/domain_dist.png",
            dpi=360, bbox_inches="tight", facecolor="white")
plt.close()
print("Done")

/var/folders/0x/3jp31_t54llb6607nqdym62c0000gn/T/ipykernel_65636/675195129.py:119: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Done


In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── PASTE YOUR DATA HERE ──────────────────────────────────────────────────────
num_domain_dist = Counter([len(s["grouped_domains"]) for s in domain_results["data"] if "grouped_domains" in s])
# ─────────────────────────────────────────────────────────────────────────────

df = pd.DataFrame({
    "domains": list(num_domain_dist.keys()),
    "count":   list(num_domain_dist.values()),
})
df = df.sort_values("domains").reset_index(drop=True)
df["domains"] = df["domains"].astype(str)

# ── Light pastel palette (one color per bar) ──────────────────────────────────
N      = len(df)
hues   = np.linspace(0, 1, N, endpoint=False)
colors = [mcolors.hsv_to_rgb([h, 0.30, 0.96]) for h in hues]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6), facecolor="white")
ax.set_facecolor("white")

sns.barplot(data=df, x="domains", y="count", palette=colors, ax=ax)

# Count + percentage labels on top of each bar
total = df["count"].sum()
for bar in ax.patches:
    count = int(bar.get_height())
    pct = count / total * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + total * 0.003,
        f"{count}\n({pct:.1f}%)",
        ha="center", va="bottom", fontsize=20, fontweight="bold", color="#444444"
    )

ax.set_xlabel("Number of domains", fontsize=24, color="#444444")
ax.set_ylabel("Number of questions", fontsize=24, color="#444444")
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(colors="#666666")
ax.tick_params(axis='x', labelsize=18)
ax.tick_params(axis='y', labelsize=18)

plt.tight_layout()
plt.savefig("../figures/num_domains_dist.pdf", dpi=360, bbox_inches="tight", facecolor="white")
plt.close()

/var/folders/0x/3jp31_t54llb6607nqdym62c0000gn/T/ipykernel_15558/2957253946.py:26: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x="domains", y="count", palette=colors, ax=ax)


In [24]:
num_domain_dist

Counter({3: 1417, 2: 458, 4: 451, 1: 61, 5: 24, 6: 2})

In [ ]:
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Data ───────────────────────────────────────────────────────────────────────
domain_counts = Counter([
    grouped_domain
    for s in domain_results["data"]
    for grouped_domain in s["grouped_domains"]
])

domain_name_mapping = {
    "Art History & Visual Culture": "Art & Culture",
    "Earth & Environmental Science": "Earth & Environment",
    "Medicine & Health Sciences": "Medicine & Health",
    "Astronomy & Space Science": "Astronomy",
    "Physical Education & Sports": "Sports",
    "Home Economics & Daily Life": "Home & Daily Life",
    "Science": "Other Sciences",
    "Design & Architecture": "Design & Arch.",
}

models = [
    ("Gemini-3.1-Pro (medium)", "../experiments/outputs/gemini-3.1-pro-preview/chgk_en_benchmark_eval_reasoning_model_en_s0_gemini-3.1-pro-preview_20260310_105704.json"),
    ("Gemini-3.1-Pro (high)", "../experiments/outputs/gemini-3.1-pro-preview/chgk_en_benchmark_eval_reasoning_model_en_s0_gemini-3.1-pro-preview_20260312_204451.json"),
    ("GPT-4.1", "../experiments/outputs/gpt-4.1/chgk_en_benchmark_eval_cot_answer_en_s0_gpt-4.1_20260228_123352.json"),
    ("Qwen3-235B-A22B-Thinking", "../experiments/outputs/qwen3-235b-a22b-thinking-2507/chgk_en_benchmark_eval_reasoning_model_en_s0_qwen3-235b-a22b-thinking-2507_20260312_203137.json"),
    ("DeepSeek-V3.2", "../experiments/outputs/deepseek-v3.2/chgk_en_benchmark_eval_reasoning_model_en_s0_deepseek-v3.2_20260315_133331.json"),
    ("GPT-5.4 (medium)", "../experiments/outputs/gpt-5.4/chgk_en_benchmark_eval_reasoning_model_en_s0_gpt-5.4_20260310_110106.json"),
]

domains = [domain for domain, counts in domain_counts.items() if counts > 30]
domains = sorted(domains, key=lambda d: domain_counts[d], reverse=True)

domain_labels = [
    f"{domain_name_mapping.get(domain, domain)} ({domain_counts[domain]})"
    for domain in domains
]

# ── Compute per-model domain accuracy vectors ─────────────────────────────────
results = {}
for model, model_path in models:
    model_results = read_json(model_path)
    model_accs = []
    for domain in domains:
        domain_sample_ids = [
            s["id"]
            for s in domain_results["data"]
            if domain in s.get("grouped_domains", [])
        ]
        domain_samples = [s for s in model_results["data"] if s["id"] in domain_sample_ids]
        domain_acc = np.mean([int(s.get("gpt-4o_judge_match", False)) for s in domain_samples])
        model_accs.append(domain_acc)
    results[model] = model_accs

# ── Single heatmap: models as rows, domains as columns ────────────────────────
heatmap_df = pd.DataFrame(
    [results[model_name] for model_name, _ in models],
    index=[model_name for model_name, _ in models],
    columns=domain_labels,
)

earthy_cmap = sns.blend_palette(
    ["#F2E8D5", "#D6B48A", "#B08968", "#7F5539", "#5E3B2E"],
    as_cmap=True,
 )
fig_width = max(18, len(domains) * 0.55)
fig_height = max(5.5, len(models) * 0.85 + 2.0)
fig, ax = plt.subplots(figsize=(fig_width, fig_height), facecolor="white")

sns.heatmap(
    heatmap_df,
    ax=ax,
    cmap=earthy_cmap,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 9},
    linewidths=0.5,
    linecolor="white",
    cbar=True,
    cbar_kws={"label": "Accuracy", "shrink": 0.85},
    vmin=0,
    vmax=1,
)

ax.set_xlabel("Domain", fontsize=11, color="#3F3A35")
ax.set_ylabel("Model", fontsize=11, color="#3F3A35")
ax.tick_params(axis="x", labelsize=8, colors="#3F3A35")
ax.tick_params(axis="y", labelsize=10, colors="#3F3A35")
plt.xticks(rotation=70, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig("../figures/domain-results.pdf", dpi=360, bbox_inches="tight", facecolor="white")
plt.close()

In [34]:
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Data ───────────────────────────────────────────────────────────────────────
domain_counts = Counter([
    grouped_domain
    for s in domain_results["data"]
    for grouped_domain in s["grouped_domains"]
])

domain_name_mapping = {
    "Art History & Visual Culture": "Art & Culture",
    "Earth & Environmental Science": "Earth & Environment",
    "Medicine & Health Sciences": "Medicine & Health",
    "Astronomy & Space Science": "Astronomy",
    "Physical Education & Sports": "Sports",
    "Home Economics & Daily Life": "Home & Daily Life",
    "Science": "Other Sciences",
    "Design & Architecture": "Design & Arch.",
}

models = [
    ("Gemini-3.1-Pro (medium)", "../experiments/outputs/gemini-3.1-pro-preview/chgk_ru_benchmark_eval_reasoning_model_ru_en_s0_gemini-3.1-pro-preview_20260310_160621.json"),
    ("Gemini-3.1-Pro (high)", "../experiments/outputs/gemini-3.1-pro-preview/chgk_ru_benchmark_eval_reasoning_model_ru_en_s0_gemini-3.1-pro-preview_20260313_232947.json"),
    ("GPT-4.1", "../experiments/outputs/gpt-4.1/chgk_ru_benchmark_eval_cot_answer_ru_en_s0_gpt-4.1_20260228_160027.json"),
    ("Qwen3-235B-A22B-Thinking", "../experiments/outputs/qwen3-235b-a22b-thinking-2507/chgk_ru_benchmark_eval_reasoning_model_ru_en_s0_qwen3-235b-a22b-thinking-2507_20260314_102839.json"),
    ("DeepSeek-V3.2", "../experiments/outputs/deepseek-v3.2/chgk_ru_benchmark_eval_reasoning_model_ru_en_s0_deepseek-v3.2_20260315_165350.json"),
    ("GPT-5.4 (medium)", "../experiments/outputs/gpt-5.4/chgk_ru_benchmark_eval_reasoning_model_ru_en_s0_gpt-5.4_20260310_132133.json"),
]

domains = [domain for domain, counts in domain_counts.items() if counts > 30]
domains = sorted(domains, key=lambda d: domain_counts[d], reverse=True)

domain_labels = [
    f"{domain_name_mapping.get(domain, domain)} ({domain_counts[domain]})"
    for domain in domains
]

# ── Compute per-model domain accuracy vectors ─────────────────────────────────
results = {}
for model, model_path in models:
    model_results = read_json(model_path)
    model_accs = []
    for domain in domains:
        domain_sample_ids = [
            s["id"]
            for s in domain_results["data"]
            if domain in s.get("grouped_domains", [])
        ]
        domain_samples = [s for s in model_results["data"] if s["id"] in domain_sample_ids]
        domain_acc = np.mean([int(s.get("gpt-4o_judge_match", False)) for s in domain_samples])
        model_accs.append(domain_acc)
    results[model] = model_accs

# ── Single heatmap: models as rows, domains as columns ────────────────────────
heatmap_df = pd.DataFrame(
    [results[model_name] for model_name, _ in models],
    index=[model_name for model_name, _ in models],
    columns=domain_labels,
)

earthy_cmap = sns.blend_palette(
    ["#F2E8D5", "#D6B48A", "#B08968", "#7F5539", "#5E3B2E"],
    as_cmap=True,
 )
fig_width = max(18, len(domains) * 0.55)
fig_height = max(5.5, len(models) * 0.85 + 2.0)
fig, ax = plt.subplots(figsize=(fig_width, fig_height), facecolor="white")

sns.heatmap(
    heatmap_df,
    ax=ax,
    cmap=earthy_cmap,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 9},
    linewidths=0.5,
    linecolor="white",
    cbar=True,
    cbar_kws={"label": "Accuracy", "shrink": 0.85},
    vmin=0,
    vmax=1,
)

ax.set_xlabel("Domain", fontsize=11, color="#3F3A35")
ax.set_ylabel("Model", fontsize=11, color="#3F3A35")
ax.tick_params(axis="x", labelsize=8, colors="#3F3A35")
ax.tick_params(axis="y", labelsize=10, colors="#3F3A35")
plt.xticks(rotation=70, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig("../figures/domain-results-ru.pdf", dpi=360, bbox_inches="tight", facecolor="white")
plt.close()